# 13 · Open the AR6 archive without downloading the world

**Time:** 35 minutes · **Audience:** undergraduate / introductory graduate GIS and Earth science.

**By the end:** Inspect actual metadata, estimate chunk costs, and distinguish samples, scenarios, workflows, and locations.

[Run in JupyterLite](https://jltobias.github.io/JupyterLite-zarr-sql-views/lite/lab/index.html?path=13_ar6_archive.ipynb) · [Earth lab](https://jltobias.github.io/JupyterLite-zarr-sql-views/lab/) · [Sources and licenses](https://jltobias.github.io/JupyterLite-zarr-sql-views/book/credits.html)

Run cells from top to bottom with **Shift+Enter**. The first kernel start downloads Python WebAssembly packages.
The bundled exercises need no data-service account. Save a copy with **File → Save and Export Notebook As** before clearing browser storage.

![Four-stage workflow: STAC discovery, Zarr reading, SQL selection, linked views](assets/pipeline.svg)

In [1]:
from course import *
import numpy as np
import matplotlib.pyplot as plt
print('Ready: Python + NumPy + local teaching data')

Ready: Python + NumPy + local teaching data


> **Data identity:** real modeled projections, not observations. AR6 v20210809; workflow `wf_1e`; 20,000 Monte Carlo samples at each decade from 2020 to 2100. Source values are millimetres of relative sea-level change from the 1995–2014 reference period; plots convert them to metres. Background land motion is included.

> **Interpretation:** a workflow distribution is not the assessed IPCC probability box (p-box). The source FAQ says `wf_1e` does not preserve time correlations: do not calculate rates, sample trajectories, or first-crossing dates. The exercise compares distributions at specified years. These data alone cannot map inundation or rank port vulnerability.

![From archive chunks to projection distributions, with flood-risk evidence kept separate](assets/ar6-workflow.svg)

## 1. Ask a small question first

We will study **Abidjan, Lagos, Bangkok, Ho Chi Minh City, and Manila**. Begin with a particular location, year, scenario, and statistic. The source's 38.42 GB gauge archive and 1.65 TB gridded archive describe many stores, not a single download needed for this lesson.

`gs://bucket/path` becomes `https://storage.googleapis.com/bucket/path` for public HTTP access. A readable URL in desktop Python still needs CORS permission for browser fetch. No Google account is needed for the bundled extracts.

In [2]:
from ar6 import load_ar6, coverage_map, coastline, gauge_summary, summary_sql, export_summary
p, years, gauge_m, grid_m = load_ar6()
print('Samples: scenario, sample, year, unique gauge =', gauge_m.shape)
print('Nearest defined grid samples: sample, year, city =', grid_m.shape)

Samples: scenario, sample, year, unique gauge = (2, 20000, 9, 4)
Nearest defined grid samples: sample, year, city = (20000, 9, 5)


In [3]:
for source in p['stores']:
    m=source['metadata']; a=m['sea_level_change/.zarray']
    print(source['partition'], source['scenario'])
    print(source['url'])
    print('Axes:',m['sea_level_change/.zattrs']['_ARRAY_DIMENSIONS'])
    print('Shape:',a['shape'],'chunks:',a['chunks'],'codec:',a['compressor'])
    print('Decoded chunk MiB:',round(np.prod(a['chunks'])*np.dtype(a['dtype']).itemsize/1024**2,2))

tide-gauges ssp126
https://storage.googleapis.com/ar6-lsl-simulations-public-standard/tide-gauges/full_sample_workflows/wf_1e/ssp126/total-workflow.zarr
Axes: ['samples', 'years', 'locations']
Shape: [20000, 9, 1012] chunks: [5000, 5, 1012] codec: {'blocksize': 0, 'clevel': 5, 'cname': 'lz4', 'id': 'blosc', 'shuffle': 1}
Decoded chunk MiB: 96.51
tide-gauges ssp585
https://storage.googleapis.com/ar6-lsl-simulations-public-standard/tide-gauges/full_sample_workflows/wf_1e/ssp585/total-workflow.zarr
Axes: ['samples', 'years', 'locations']
Shape: [20000, 9, 1012] chunks: [5000, 5, 1012] codec: {'blocksize': 0, 'clevel': 5, 'cname': 'lz4', 'id': 'blosc', 'shuffle': 1}
Decoded chunk MiB: 96.51
gridded ssp585
https://storage.googleapis.com/ar6-lsl-simulations-public-standard/gridded/full_sample_workflows/wf_1e/ssp585/total-workflow.zarr
Axes: ['samples', 'years', 'lat', 'lon']
Shape: [20000, 9, 181, 360] chunks: [5000, 5, 30, 30] codec: {'blocksize': 0, 'clevel': 5, 'cname': 'lz4', 'id': 'blos

## 2. Calculate the cost before reading values

One gauge chunk spans **5,000 samples × 5 years × 1,012 locations**. A point selection does not make its compressed chunk smaller. The gridded store instead has `(sample, year, latitude, longitude)` axes, with **30 × 30** spatial chunks. Inspect axes: do not substitute a sample axis for time.

Selecting all 20,000 samples and nine years touches eight chunks per gauge store. The five small gridded windows intersect four spatial tiles, each with eight sample/year chunks. These chunks were downloaded once during preparation. Students read only the derivative files.

In [4]:
objects=p['source_objects']
chunks=[r for r in objects if '/sea_level_change/' in r['url']]
print('Value chunks actually fetched:',len(chunks))
print('One-time source transfer MB:',round(p['unique_source_bytes']/1e6,1))
print('Bundled statistical artifacts MB:',round(sum(a['bytes'] for a in p['artifacts'].values())/1e6,2))
print('Source data retained:',p['samples'],'samples per selected location/year')

Value chunks actually fetched: 48
One-time source transfer MB: 1340.9
Bundled statistical artifacts MB: 4.48
Source data retained: 20000 samples per selected location/year


## 3. Trace a byte to a source

The provenance records source URLs, object generation IDs where supplied, retrieval timestamps, SHA-256 hashes, coordinate choices, and transformations. A full refresh is an **instructor/desktop** operation using `python scripts/prepare_ar6.py --download`. It limits each decoded chunk to 128 MiB and new downloads to 2 GiB per run, and reuses verified cached objects. Normal builds and notebooks never fetch that archive.

The browser-facing cubes contain per-year medians, converted to uncompressed Zarr v2. The NPZ files preserve all samples for the selected locations. Reducing samples to quantiles and selecting locations are stated modifications under the data license.

In [5]:
record=chunks[0]
print(json.dumps(record,indent=2))
print('A source variable:',p['stores'][0]['metadata']['sea_level_change/.zattrs'])

{
  "url": "https://storage.googleapis.com/ar6-lsl-simulations-public-standard/tide-gauges/full_sample_workflows/wf_1e/ssp126/total-workflow.zarr/sea_level_change/0.0.0",
  "bytes": 41792844,
  "sha256": "aa6406950303bec67bb07671257c2b2d11f5e0f5586856a7cfeb67fb25597003",
  "retrieved_utc": "2026-10-13T16:29:56.484097+00:00",
  "etag": "\"195a3bbd876af63df68364508a323613\"",
  "generation": "1654491400230294",
  "cors_allow_origin": null
}
A source variable: {'_ARRAY_DIMENSIONS': ['samples', 'years', 'locations'], 'coordinates': 'lon lat', 'units': 'mm'}


## 4. Optional live metadata experiment

This requests only one `.zmetadata` file (about 8 KB), never a value chunk. Leave `LIVE=False` for the recorded, reproducible lesson. Public Google Storage responses inspected during preparation did not provide an `Access-Control-Allow-Origin` header for the course origin. A browser failure therefore does not establish that a store is absent. The code reports the failure explicitly; it never substitutes fabricated values.

In [6]:
LIVE=False
url=p['stores'][0]['url']+'/.zmetadata'
if LIVE:
    try:
        if __import__('sys').platform=='emscripten':
            from pyodide.http import pyfetch
            response=await pyfetch(url)
            if not response.ok: raise RuntimeError(f'HTTP {response.status}')
            live=await response.json()
        else:
            import urllib.request
            with urllib.request.urlopen(url,timeout=30) as response: live=json.load(response)
        print('LIVE metadata axes:',live['metadata']['sea_level_change/.zattrs']['_ARRAY_DIMENSIONS'])
    except Exception as exc:
        print('LIVE METADATA UNAVAILABLE:',type(exc).__name__,str(exc))
        print('The recorded metadata above remains usable; no live sample values were requested.')
else:
    print('RECORDED METADATA MODE: no archive network requests.')

RECORDED METADATA MODE: no archive network requests.


## 5. Find the question's proper dataset

| Question | Suitable evidence |
|---|---|
| Distribution at one location/year under one workflow | The supplied Monte Carlo samples |
| Assessed IPCC range across workflows | Authors' confidence/p-box products or [NASA's projection tool](https://sealevel.nasa.gov/ipcc-ar6-sea-level-projection-tool) |
| Rate or first-crossing date | Appropriate time-correlated workflow or supplied milestone products, not `wf_1e` paths |
| Which port assets flood? | Datum-aligned elevation, tides/surge/waves, hydraulic connectivity, defenses and asset exposure |

The free public **totals** used here include background land motion. No requester-pays bucket or no-VLM product is used.

## Try it yourself

Estimate chunks needed for one gauge in 2050 versus one gauge at all nine years. Explain why SQL filtering after loading does not change this cost.

## Check your reasoning

With all samples, 2050 touches four chunks; nine years touch eight. Both include all gauge locations within each fetched chunk. Source selection precedes decoding; SQL acts on the extracted table.

## Evidence journal

Write four sentences: **what I observed; how I calculated it; what this cannot establish; what evidence I need next**.
For any figure you reuse, retain its units, date or scenario, data origin, transformations, and attribution.

## Sources and credit

- [Fox-Kemper et al. (2021), IPCC AR6 WG1 Chapter 9: Ocean, Cryosphere and Sea Level Change](https://doi.org/10.1017/9781009157896.011).
- [Kopp et al. (2023), FACTS v1.0, Geoscientific Model Development 16, 7461–7489](https://doi.org/10.5194/gmd-16-7461-2023).
- [Garner et al. (2021), IPCC AR6 Sea Level Projections, version 20210809](https://doi.org/10.5281/zenodo.5914709), accessed 2026-10-13. Modified extracts under [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/).
- [Rutgers-ESSP source guide and workflow FAQ](https://github.com/Rutgers-ESSP/IPCC-AR6-Sea-Level-Projections/tree/8ee75128d8f6d9a8ba9825f47c92d11955d0651e).
- [Exact source objects, checksums, transformations and location distances](https://jltobias.github.io/JupyterLite-zarr-sql-views/data/ar6-provenance.json).
- We thank the projection authors for developing and making the sea-level rise projections available, multiple funding agencies for supporting the development of the projections, and the NASA Sea Level Change Team for developing and hosting the IPCC AR6 Sea Level Projection Tool.

Teaching adaptation of [dzole0311/zarr-sql-views](https://github.com/dzole0311/zarr-sql-views), ISC.
Original teaching code and prose: JupyterLite-zarr-sql-views contributors, ISC. Data keep their separate licenses.

In [7]:
print('LESSON COMPLETE: 13_ar6_archive')

LESSON COMPLETE: 13_ar6_archive
